# 外部知识抓取队列

目标：把 GitHub、论文、Twitter/X、arXiv 等外部知识先整理成结构化研究卡和机器可读 backlog，再进入 factor/model/objective/analysis 验证。skill 只保留方法论，不保存业务状态。

In [1]:
from pathlib import Path
import json
import sys

REPO_ROOT = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "AGENTS.md").exists() and (path / "src").exists())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

from src.core import KnowledgeSource, write_knowledge_backlog, write_knowledge_queue

OUTPUT_DIR = REPO_ROOT / "research" / "knowledge_ingest"
RETRIEVED_AT = "2026-06-19"
sources = [
    KnowledgeSource(
        source_id="arxiv:2504.18600",
        source_type="arxiv",
        title="QuantBench: Benchmarking AI Methods for Quantitative Investment",
        url="https://arxiv.org/abs/2504.18600",
        summary="Benchmark reference for a standardized full-pipeline quant workflow, model matrix coverage, continual-learning stress, relation modeling, and overfitting control.",
        license_note="只保存公开 URL 和自有摘要；不复制未授权全文。",
        retrieved_at=RETRIEVED_AT,
        access_status="metadata_accessible",
    ),
    KnowledgeSource(
        source_id="github:microsoft/qlib",
        source_type="github",
        title="Microsoft Qlib",
        url="https://github.com/microsoft/qlib",
        summary="Qlib is an AI-oriented quantitative investment platform reference for research workflow, recorder/analyzer style artifacts, and production handoff boundaries.",
        license_note="进入实现前复核 upstream license；不 vendoring 代码。",
        retrieved_at=RETRIEVED_AT,
        access_status="repository_accessible",
    ),
    KnowledgeSource(
        source_id="github:thuml/Time-Series-Library",
        source_type="github",
        title="Time-Series-Library",
        url="https://github.com/thuml/Time-Series-Library",
        summary="Reference implementation hub for long-series forecasting families such as DLinear, Autoformer, FEDformer, and PatchTST; useful for adapter-fidelity backlog only after license review.",
        license_note="进入实现前复核 upstream license；只记录 URL 和摘要。",
        retrieved_at=RETRIEVED_AT,
        access_status="repository_accessible",
    ),
    KnowledgeSource(
        source_id="arxiv:2211.14730",
        source_type="arxiv",
        title="A Time Series is Worth 64 Words: Long-term Forecasting with Transformers",
        url="https://arxiv.org/abs/2211.14730",
        summary="PatchTST paper reference for patching and channel-independence ideas that should be validated through the existing patch_tst_mse adapter contract.",
        license_note="arXiv 元数据和自有摘要；实现代码 license 单独复核。",
        retrieved_at=RETRIEVED_AT,
        access_status="metadata_accessible",
    ),
    KnowledgeSource(
        source_id="arxiv:2205.13504",
        source_type="paper",
        title="Are Transformers Effective for Time Series Forecasting?",
        url="https://arxiv.org/abs/2205.13504",
        summary="DLinear-style decomposition-linear baseline reference; useful for checking whether simple sequence baselines beat complex adapters in SOXL regimes.",
        license_note="arXiv 元数据和自有摘要；不复制全文。",
        retrieved_at=RETRIEVED_AT,
        access_status="metadata_accessible",
    ),
    KnowledgeSource(
        source_id="arxiv:2106.13008",
        source_type="paper",
        title="Autoformer: Decomposition Transformers with Auto-Correlation for Long-Term Series Forecasting",
        url="https://arxiv.org/abs/2106.13008",
        summary="Autoformer decomposition and auto-correlation reference for the autoformer_mse adapter fidelity backlog.",
        license_note="arXiv 元数据和自有摘要；实现代码 license 单独复核。",
        retrieved_at=RETRIEVED_AT,
        access_status="metadata_accessible",
    ),
    KnowledgeSource(
        source_id="arxiv:2201.12740",
        source_type="paper",
        title="FEDformer: Frequency Enhanced Decomposed Transformer for Long-term Series Forecasting",
        url="https://arxiv.org/abs/2201.12740",
        summary="FEDformer frequency/decomposition reference for the fedformer_mse adapter fidelity backlog.",
        license_note="arXiv 元数据和自有摘要；实现代码 license 单独复核。",
        retrieved_at=RETRIEVED_AT,
        access_status="metadata_accessible",
    ),
    KnowledgeSource(
        source_id="github:quantopian/alphalens",
        source_type="github",
        title="Alphalens factor performance analysis",
        url="https://github.com/quantopian/alphalens",
        summary="GitHub factor-analysis reference for returns analysis, information coefficient analysis, turnover analysis, and grouped analysis; useful only as a src.factor diagnostics hypothesis.",
        license_note="GitHub README shows Apache-2.0 license；进入实现前仍需复核 upstream license；不 vendoring 代码。",
        retrieved_at=RETRIEVED_AT,
        access_status="repository_accessible",
    ),
    KnowledgeSource(
        source_id="x:regime-aware-quant-search",
        source_type="x",
        title="Regime-aware quant evaluation search",
        url="https://x.com/search?q=regime%20aware%20quant%20evaluation",
        summary="Social-source idea placeholder for volatility/drawdown regime review; only usable after it becomes a reproducible src.analysis window/regime test.",
        license_note="idea-only；不复制帖子正文；不作为事实来源。",
        retrieved_at=RETRIEVED_AT,
        access_status="search_placeholder",
    ),
]
queue_result = write_knowledge_queue(OUTPUT_DIR, sources)
backlog_result = write_knowledge_backlog(OUTPUT_DIR, sources)
backlog_payload = json.loads((OUTPUT_DIR / "knowledge_backlog.json").read_text(encoding="utf-8"))
summary = {
    "schema_version": "research.knowledge_ingest.v1",
    "status": "completed",
    "queue_result": queue_result,
    "backlog_result": backlog_result,
    "source_count": backlog_payload["summary"]["source_count"],
    "target_subsystems": backlog_payload["summary"]["target_subsystems"],
    "candidate_types": backlog_payload["summary"]["candidate_types"],
    "top_candidates": backlog_payload["summary"]["top_candidates"],
    "promotion_gate_statuses": sorted({item["promotion_gate_status"] for item in backlog_payload["items"]}),
    "required_evidence_counts": sorted({item["required_evidence_count"] for item in backlog_payload["items"]}),
    "retrieved_at_values": sorted({item["retrieved_at"] for item in backlog_payload["items"]}),
    "usage_scopes": sorted({item["usage_scope"] for item in backlog_payload["items"]}),
    "source_reliabilities": sorted({item["source_reliability"] for item in backlog_payload["items"]}),
    "artifacts": {
        "knowledge_queue": "knowledge_queue.json",
        "knowledge_backlog": "knowledge_backlog.json",
        "knowledge_backlog_csv": "knowledge_backlog.csv",
        "knowledge_backlog_review": "knowledge_backlog_review.md",
    },
}
(OUTPUT_DIR / "knowledge_ingest_summary.json").write_text(json.dumps(summary, ensure_ascii=False, indent=2), encoding="utf-8")
summary


{'schema_version': 'research.knowledge_ingest.v1',
 'status': 'completed',
 'queue_result': {'written': 9, 'deduped': 0},
 'backlog_result': {'written': 9, 'deduped': 0},
 'source_count': 9,
 'target_subsystems': ['src/analysis', 'src/factor', 'src/model'],
 'candidate_types': ['analysis_extension',
  'factor_pipeline_reference',
  'method_signal',
  'model_matrix_reference',
  'model_replacement'],
 'top_candidates': ['autoformer_mse',
  'fedformer_mse',
  'dlinear_mse',
  'patch_tst_mse',
  'quantbench_model_matrix'],
 'promotion_gate_statuses': ['blocked_pending_research'],
 'required_evidence_counts': [5],
 'retrieved_at_values': ['2026-06-19'],
 'usage_scopes': ['hypothesis_and_adapter_reference', 'idea_only'],
 'source_reliabilities': ['primary_metadata',
  'primary_repository_metadata',
  'social_unverified'],
 'artifacts': {'knowledge_queue': 'knowledge_queue.json',
  'knowledge_backlog': 'knowledge_backlog.json',
  'knowledge_backlog_csv': 'knowledge_backlog.csv',
  'knowledge

## 中文结论

- 做了什么：把 arXiv 论文、GitHub 项目和 X/Twitter 搜索入口统一转成 `knowledge_queue.json` 与 `knowledge_backlog.json/csv`，只保存 URL、自有摘要和结构化假设，并记录 retrieved_at、access_status、usage_scope 和 source_reliability。
- 核心指标：本流程记录 source_count、target_subsystems、candidate_types、top_candidates、promotion_gate_statuses、required_evidence_counts、usage_scopes 和 source_reliabilities，不评估模型收益。
- regime 差异：社交媒体来源仅作为 `idea_only` / `social_unverified`，必须进入 `src/analysis` 的 regime/window 对比后才有价值。
- 是否晋升：否；这里只是 research ingest 入口，不直接晋升到 production，也不把项目状态写入 skill。
- 风险：所有 GitHub/论文实现都需要 license gate；所有外部假设都需要 research notebook、聚焦测试、manifest/export 和中文结论后才能进入 `src/`。
- 下一步：按 `knowledge_backlog.csv` 的 priority_rank 逐项建立模型 fidelity、qlib analyzer、regime review 的验证 notebook。